In [ ]:
# Run this to verify your model paths and structure
from pyspark.ml import PipelineModel
import os

# Check if path exists
model_path = "/content/drive/MyDrive/Bda/saved_models/20251109_063633/random_forest"
print(f"Path exists: {os.path.exists(model_path)}")
print(f"Contents: {os.listdir(model_path) if os.path.exists(model_path) else 'Path not found'}")

# Try loading
try:
    model = PipelineModel.load(model_path)
    print("Model loaded successfully!")
    print(f"Stages: {[type(stage).__name__ for stage in model.stages]}")
except Exception as e:
    print(f"Load error: {e}")

Path exists: True
Contents: ['metadata', 'treesMetadata', 'data']
Load error: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.ml.classification import RandomForestClassificationModel, GBTClassificationModel, LogisticRegressionModel, LinearSVC
import os

# Initialize Spark session
@st.cache_resource
def init_spark():
    return SparkSession.builder \
        .appName("StreamlitML") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .getOrCreate()

# Page configuration
st.set_page_config(
    page_title="Spark ML Model Deployment",
    page_icon="🤖",
    layout="wide"
)

# Title and description
st.title("🤖 Spark ML Model Deployment")
st.markdown("Load and use your saved Spark ML models for predictions")

# Initialize Spark
spark = init_spark()

# Sidebar for model selection
st.sidebar.header("Model Configuration")

# Function to load Spark ML models
@st.cache_resource
def load_spark_model(model_path, model_type):
    try:
        if model_type == "Random Forest":
            model = RandomForestClassificationModel.load(model_path)
        elif model_type == "Gradient Boosting":
            model = GBTClassificationModel.load(model_path)
        elif model_type == "Logistic Regression":
            model = LogisticRegressionModel.load(model_path)
        elif model_type == "SVM":
            model = LinearSVC.load(model_path)
        else:
            st.sidebar.error(f"❌ Unknown model type: {model_type}")
            return None
        st.sidebar.success(f"✅ Model loaded successfully from {model_path}!")
        return model
    except Exception as e:
        st.sidebar.error(f"❌ Failed to load model from {model_path}: {str(e)}")
        return None

# Model selection - adjust paths based on your actual structure
model_options = {
    "Random Forest": "/content/drive/MyDrive/Bda/saved_models/20251109_063633/random_forest",
    "Gradient Boosting": "/content/drive/MyDrive/Bda/saved_models/20251109_063633/gradient_boosting",
    "Logistic Regression": "/content/drive/MyDrive/Bda/saved_models/20251109_063633/logistic_regression",
    "SVM": "/content/drive/MyDrive/Bda/saved_models/20251030_065438/svm"
}

selected_model_name = st.sidebar.selectbox(
    "Choose Model",
    list(model_options.keys())
)

# Load selected model
model_path = model_options[selected_model_name]
model = load_spark_model(model_path, selected_model_name)

# Main content area
tab1, tab2, tab3 = st.tabs(["📊 Manual Input", "📁 File Upload", "📈 Model Info"])

with tab1:
    st.header("Manual Data Input")

    # Create sample input fields - ADJUST THESE BASED ON YOUR ACTUAL FEATURES
    st.subheader("Enter Feature Values")

    col1, col2, col3 = st.columns(3)

    with col1:
        # Example features - replace with your actual feature names
        feature1 = st.number_input("Feature 1", value=0.0, format="%.6f")
        feature2 = st.number_input("Feature 2", value=0.0, format="%.6f")

    with col2:
        feature3 = st.number_input("Feature 3", value=0.0, format="%.6f")
        feature4 = st.number_input("Feature 4", value=0.0, format="%.6f")

    with col3:
        feature5 = st.number_input("Feature 5", value=0.0, format="%.6f")
        feature6 = st.number_input("Feature 6", value=0.0, format="%.6f")

    # Create pandas DataFrame
    input_data_pd = pd.DataFrame({
        'feature1': [feature1],
        'feature2': [feature2],
        'feature3': [feature3],
        'feature4': [feature4],
        'feature5': [feature5],
        'feature6': [feature6]
        # Add more features as needed
    })

    if st.button("Predict", type="primary"):
        if model is not None:
            try:
                # Convert to Spark DataFrame
                input_data_spark = spark.createDataFrame(input_data_pd)

                # Make prediction
                predictions = model.transform(input_data_spark)

                # Collect results
                results = predictions.toPandas()

                # Display results
                st.success("✅ Prediction completed successfully!")

                st.subheader("Prediction Results")
                st.dataframe(results)

                # Extract prediction column (usually 'prediction' in Spark ML)
                prediction_cols = [col for col in results.columns if 'prediction' in col.lower()]
                if prediction_cols:
                    prediction_value = results[prediction_cols[0]].iloc[0]
                    st.metric("Final Prediction", prediction_value)

            except Exception as e:
                st.error(f"Prediction error: {str(e)}")
        else:
            st.error("Model not loaded properly!")

with tab2:
    st.header("Upload Data File")

    uploaded_file = st.file_uploader(
        "Upload CSV file for predictions",
        type=['csv'],
        help="Upload a CSV file with the same features used during training"
    )

    if uploaded_file is not None:
        try:
            # Read the uploaded file
            data_pd = pd.read_csv(uploaded_file)
            st.subheader("Uploaded Data Preview")
            st.dataframe(data_pd.head())

            if st.button("Predict on Uploaded Data"):
                if model is not None:
                    # Convert to Spark DataFrame
                    data_spark = spark.createDataFrame(data_pd)

                    # Make predictions
                    predictions = model.transform(data_spark)

                    # Convert back to pandas for display
                    results = predictions.toPandas()

                    st.subheader("Prediction Results")
                    st.dataframe(results)

                    # Download results
                    csv = results.to_csv(index=False)
                    st.download_button(
                        label="Download Predictions as CSV",
                        data=csv,
                        file_name=f"predictions_{selected_model_name.replace(' ', '_').lower()}.csv",
                        mime="text/csv"
                    )
                else:
                    st.error("Model not loaded properly!")

        except Exception as e:
            st.error(f"Error processing file: {str(e)}")

with tab3:
    st.header("Model Information")

    if model is not None:
        st.subheader(f"{selected_model_name} Details")

        # Display model stages
        st.write("**Pipeline Stages:**")
        try:
             for i, stage in enumerate(model.stages):
                 st.write(f"{i+1}. {type(stage).__name__}")
        except:
             st.write("Model is not a PipelineModel and stages cannot be displayed in this way.")


        # Try to display model parameters
        st.write("**Model Parameters:**")
        try:
            # For classification models
            if hasattr(model, 'getNumTrees'):
                st.write(f"Number of Trees: {model.getNumTrees()}")
            if hasattr(model, 'getMaxDepth'):
                st.write(f"Max Depth: {model.getMaxDepth()}")
        except:
            st.write("Parameter details not available")
    else:
        st.warning("Please select and load a model first")

# Footer
st.markdown("---")
st.markdown("Built with Streamlit • Spark ML Models")

# Add instructions
with st.expander("ℹ️ Setup Instructions"):
    st.markdown("""
    **Requirements:**
    - PySpark installed: `pip install pyspark`
    - Your Spark ML models should be in the correct directory structure
    - Adjust feature names in the code to match your actual model features

    **Common Issues:**
    - Ensure model paths are correct
    - Check that all model files are present
    - Verify feature names match training data
    """)

Writing app.py


In [ ]:
!pip install streamlit pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 131.5 MB/s eta 0:00:00


In [ ]:
!pip install pyngrok


In [ ]:
%%writefile apps.py
import streamlit as st
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.ml.classification import RandomForestClassificationModel, GBTClassificationModel, LogisticRegressionModel
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.types import StructType, StructField, DoubleType
import os

# Initialize Spark session
@st.cache_resource
def init_spark():
    return SparkSession.builder \
        .appName("StreamlitML") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .master("local[*]") \
        .getOrCreate()

# Page configuration
st.set_page_config(
    page_title="Spark ML Model Deployment",
    page_icon="🤖",
    layout="wide"
)

# Title and description
st.title("🤖 Credit Card Fraud Detection")
st.markdown("Load and use your saved Spark ML models for fraud prediction")

# Initialize Spark
spark = init_spark()

# Sidebar for model selection
st.sidebar.header("Model Configuration")

# Function to detect model type
def detect_model_type(model_path):
    """Detect the actual model type from metadata"""
    try:
        metadata_path = os.path.join(model_path, "metadata")
        if os.path.exists(metadata_path):
            with open(metadata_path, 'r') as f:
                import json
                metadata = json.load(f)
                class_name = metadata.get('class', '')

                if 'GBTClassificationModel' in class_name:
                    return 'Gradient Boosting'
                elif 'RandomForestClassificationModel' in class_name:
                    return 'Random Forest'
                elif 'LogisticRegressionModel' in class_name:
                    return 'Logistic Regression'
                elif 'PipelineModel' in class_name:
                    return 'Pipeline'
        return 'Unknown'
    except Exception as e:
        st.sidebar.warning(f"Could not detect model type: {e}")
        return 'Unknown'

# Function to load Spark ML models
@st.cache_resource
def load_spark_model(model_path, model_type):
    try:
        # First check if path exists
        if not os.path.exists(model_path):
            st.sidebar.error(f"❌ Model path does not exist: {model_path}")
            return None

        # Detect actual model type
        actual_type = detect_model_type(model_path)
        if actual_type != model_type and actual_type != 'Unknown':
            st.sidebar.warning(f"⚠️ Model type mismatch: Expected {model_type}, found {actual_type}")
            model_type = actual_type  # Use the detected type

        # Load based on type
        if model_type == "Random Forest":
            model = RandomForestClassificationModel.load(model_path)
        elif model_type == "Gradient Boosting":
            model = GBTClassificationModel.load(model_path)
        elif model_type == "Logistic Regression":
            model = LogisticRegressionModel.load(model_path)
        elif model_type == "Pipeline":
            model = PipelineModel.load(model_path)
        else:
            st.sidebar.error(f"❌ Unsupported model type: {model_type}")
            return None

        st.sidebar.success(f"✅ {model_type} model loaded successfully!")
        return model

    except Exception as e:
        st.sidebar.error(f"❌ Failed to load model: {str(e)}")
        return None

# Model selection
model_options = {
    "Random Forest": "/content/drive/MyDrive/Bda/saved_models/20251109_063633/random_forest",
    "Gradient Boosting": "/content/drive/MyDrive/Bda/saved_models/20251109_063633/gradient_boosting",
    "Logistic Regression": "/content/drive/MyDrive/Bda/saved_models/20251109_063633/logistic_regression"
}

selected_model_name = st.sidebar.selectbox(
    "Choose Model",
    list(model_options.keys())
)

# Model path input for custom paths
custom_path = st.sidebar.text_input(
    "Or enter custom model path:",
    value=model_options[selected_model_name]
)

# Load selected model
model_path = custom_path if custom_path else model_options[selected_model_name]
model = load_spark_model(model_path, selected_model_name)

# Define feature columns for credit card fraud detection
FEATURE_COLUMNS = [f'V{i}' for i in range(1, 29)] + ['Amount']
ALL_COLUMNS = ['Time'] + FEATURE_COLUMNS + ['Class']

def prepare_features(input_df):
    """Prepare features for the model by creating the 'features' vector"""
    try:
        # Convert to Spark DataFrame
        spark_df = spark.createDataFrame(input_df)

        # Check if we need to create the features vector
        if 'features' not in spark_df.columns:
            # Create VectorAssembler to combine features
            assembler = VectorAssembler(
                inputCols=FEATURE_COLUMNS,
                outputCol="features",
                handleInvalid="skip"
            )

            # Transform the data
            prepared_df = assembler.transform(spark_df)
            return prepared_df
        else:
            # Features already exist
            return spark_df

    except Exception as e:
        st.error(f"❌ Feature preparation error: {str(e)}")
        return None

# Main content area
tab1, tab2, tab3 = st.tabs(["📊 Manual Input", "📁 File Upload", "📈 Model Info"])

with tab1:
    st.header("Manual Transaction Input")
    st.subheader("Enter Transaction Features")

    # Create columns for better organization
    col1, col2, col3, col4 = st.columns(4)

    # Time and Amount
    with col1:
        st.subheader("Basic Info")
        Time = st.number_input("Time", value=0.0, format="%.1f")
        Amount = st.number_input("Amount", value=0.0, format="%.6f")

    # V1-V7
    with col2:
        st.subheader("Features V1-V7")
        V1 = st.number_input("V1", value=0.0, format="%.6f")
        V2 = st.number_input("V2", value=0.0, format="%.6f")
        V3 = st.number_input("V3", value=0.0, format="%.6f")
        V4 = st.number_input("V4", value=0.0, format="%.6f")
        V5 = st.number_input("V5", value=0.0, format="%.6f")
        V6 = st.number_input("V6", value=0.0, format="%.6f")
        V7 = st.number_input("V7", value=0.0, format="%.6f")

    # V8-V14
    with col3:
        st.subheader("Features V8-V14")
        V8 = st.number_input("V8", value=0.0, format="%.6f")
        V9 = st.number_input("V9", value=0.0, format="%.6f")
        V10 = st.number_input("V10", value=0.0, format="%.6f")
        V11 = st.number_input("V11", value=0.0, format="%.6f")
        V12 = st.number_input("V12", value=0.0, format="%.6f")
        V13 = st.number_input("V13", value=0.0, format="%.6f")
        V14 = st.number_input("V14", value=0.0, format="%.6f")

    # V15-V21
    with col4:
        st.subheader("Features V15-V21")
        V15 = st.number_input("V15", value=0.0, format="%.6f")
        V16 = st.number_input("V16", value=0.0, format="%.6f")
        V17 = st.number_input("V17", value=0.0, format="%.6f")
        V18 = st.number_input("V18", value=0.0, format="%.6f")
        V19 = st.number_input("V19", value=0.0, format="%.6f")
        V20 = st.number_input("V20", value=0.0, format="%.6f")
        V21 = st.number_input("V21", value=0.0, format="%.6f")

    # V22-V28 in a new row
    col5, col6, col7 = st.columns(3)
    with col5:
        st.subheader("Features V22-V24")
        V22 = st.number_input("V22", value=0.0, format="%.6f")
        V23 = st.number_input("V23", value=0.0, format="%.6f")
        V24 = st.number_input("V24", value=0.0, format="%.6f")
    with col6:
        st.subheader("Features V25-V27")
        V25 = st.number_input("V25", value=0.0, format="%.6f")
        V26 = st.number_input("V26", value=0.0, format="%.6f")
        V27 = st.number_input("V27", value=0.0, format="%.6f")
    with col7:
        st.subheader("Feature V28")
        V28 = st.number_input("V28", value=0.0, format="%.6f")

    # Create pandas DataFrame with all expected columns
    input_data_pd = pd.DataFrame({
        'Time': [Time],
        'V1': [V1], 'V2': [V2], 'V3': [V3], 'V4': [V4], 'V5': [V5], 'V6': [V6], 'V7': [V7],
        'V8': [V8], 'V9': [V9], 'V10': [V10], 'V11': [V11], 'V12': [V12], 'V13': [V13], 'V14': [V14],
        'V15': [V15], 'V16': [V16], 'V17': [V17], 'V18': [V18], 'V19': [V19], 'V20': [V20], 'V21': [V21],
        'V22': [V22], 'V23': [V23], 'V24': [V24], 'V25': [V25], 'V26': [V26], 'V27': [V27], 'V28': [V28],
        'Amount': [Amount]
        # Note: 'Class' is the target variable, not needed for prediction
    })

    if st.button("Predict Fraud", type="primary"):
        if model is not None:
            try:
                # Prepare features (create 'features' vector)
                prepared_data = prepare_features(input_data_pd)

                if prepared_data is not None:
                    # Make prediction
                    predictions = model.transform(prepared_data)

                    # Collect results
                    results = predictions.toPandas()

                    # Display results
                    st.success("✅ Prediction completed successfully!")
                    st.subheader("Fraud Detection Results")

                    # Show prediction
                    prediction_value = results['prediction'].iloc[0]
                    fraud_probability = results['probability'].iloc[0][1] if 'probability' in results.columns else None

                    # Display result with appropriate styling
                    if prediction_value == 1:
                        st.error(f"🚨 **FRAUD DETECTED!** (Class: {int(prediction_value)})")
                        if fraud_probability is not None:
                            st.metric("Fraud Probability", f"{fraud_probability:.4f}")
                    else:
                        st.success(f"✅ **Legitimate Transaction** (Class: {int(prediction_value)})")
                        if fraud_probability is not None:
                            st.metric("Fraud Probability", f"{fraud_probability:.4f}")

                    # Show detailed results
                    with st.expander("View Detailed Results"):
                        st.dataframe(results)

            except Exception as e:
                st.error(f"❌ Prediction error: {str(e)}")
                st.info("💡 Check that all feature columns are present and correctly named.")
        else:
            st.error("❌ Model not loaded properly!")

with tab2:
    st.header("Upload Transaction Data File")
    st.info("Upload a CSV file with columns: Time, V1-V28, Amount")

    uploaded_file = st.file_uploader(
        "Upload CSV file for batch fraud detection",
        type=['csv'],
        help="Upload a CSV file with credit card transaction data"
    )

    if uploaded_file is not None:
        try:
            # Read the uploaded file
            data_pd = pd.read_csv(uploaded_file)
            st.subheader("Uploaded Data Preview")
            st.dataframe(data_pd.head())
            st.write(f"Data shape: {data_pd.shape}")

            # Check if required columns are present
            missing_columns = [col for col in FEATURE_COLUMNS if col not in data_pd.columns]
            if missing_columns:
                st.error(f"❌ Missing required columns: {missing_columns}")
            else:
                if st.button("Detect Fraud in Uploaded Data"):
                    if model is not None:
                        with st.spinner("Processing transactions..."):
                            # Prepare features
                            prepared_data = prepare_features(data_pd)

                            if prepared_data is not None:
                                # Make predictions
                                predictions = model.transform(prepared_data)

                                # Convert back to pandas for display
                                results = predictions.toPandas()

                                st.success(f"✅ Fraud detection completed for {len(results)} transactions!")

                                # Summary statistics
                                fraud_count = results['prediction'].sum()
                                total_count = len(results)
                                fraud_percentage = (fraud_count / total_count) * 100

                                col1, col2, col3 = st.columns(3)
                                with col1:
                                    st.metric("Total Transactions", total_count)
                                with col2:
                                    st.metric("Fraud Detected", int(fraud_count))
                                with col3:
                                    st.metric("Fraud Rate", f"{fraud_percentage:.2f}%")

                                st.subheader("Prediction Results")
                                st.dataframe(results)

                                # Download results
                                csv = results.to_csv(index=False)
                                st.download_button(
                                    label="📥 Download Predictions as CSV",
                                    data=csv,
                                    file_name=f"fraud_predictions_{selected_model_name.replace(' ', '_').lower()}.csv",
                                    mime="text/csv"
                                )
                    else:
                        st.error("❌ Model not loaded properly!")

        except Exception as e:
            st.error(f"❌ Error processing file: {str(e)}")

with tab3:
    st.header("Model Information")

    if model is not None:
        st.subheader(f"{selected_model_name} Details")
        st.write(f"**Model Path:** `{model_path}`")

        # Display model type information
        detected_type = detect_model_type(model_path)
        st.write(f"**Detected Type:** {detected_type}")

        # Display feature information
        st.write("**Expected Features:**")
        st.write(f"Using {len(FEATURE_COLUMNS)} features: V1-V28 + Amount")

        # Try to display model-specific information
        try:
            if hasattr(model, 'trees') and model.trees is not None:
                st.write(f"**Number of Trees:** {len(model.trees)}")

            if hasattr(model, 'numFeatures'):
                st.write(f"**Number of Features:** {model.numFeatures}")

            if hasattr(model, 'getNumTrees'):
                st.write(f"**Number of Trees:** {model.getNumTrees()}")

        except Exception as e:
            st.write(f"**Model Class:** {type(model).__name__}")

        # For Pipeline models, show stages
        if hasattr(model, 'stages'):
            st.write("**Pipeline Stages:**")
            for i, stage in enumerate(model.stages):
                st.write(f"{i+1}. {type(stage).__name__}")

    else:
        st.warning("⚠️ Please select and load a model first")

# Footer
st.markdown("---")
st.markdown("**Built with Streamlit • Spark ML • Credit Card Fraud Detection**")

# Add instructions
with st.expander("ℹ️ Data Format Instructions"):
    st.markdown("""
    **Expected CSV Format:**
    - **Time**: Number of seconds elapsed between this transaction and the first transaction
    - **V1-V28**: Principal components obtained from PCA (anonymized features)
    - **Amount**: Transaction amount
    - **Class** (optional): Target variable (1 = Fraud, 0 = Legitimate) - not needed for prediction

    **Required Columns for Prediction:**
    ```
    Time, V1, V2, V3, V4, V5, V6, V7, V8, V9, V10, V11, V12, V13, V14, V15,
    V16, V17, V18, V19, V20, V21, V22, V23, V24, V25, V26, V27, V28, Amount
    ```

    **Model Output:**
    - **prediction**: 0 = Legitimate, 1 = Fraud
    - **probability**: Probability scores for each class
    - **rawPrediction**: Raw model outputs (if available)
    """)

Writing apps.py


In [ ]:
# setup_ngrok.py
from pyngrok import ngrok, conf
import subprocess
import threading
import time
import os
import signal
import sys

# Your ngrok authtoken (get from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTHTOKEN = "your_authentication_token"
STREAMLIT_APP = "/content/apps.py"  # Change to your Streamlit app filename

def setup_ngrok():
    """Setup ngrok authentication and tunnel"""
    try:
        # Set auth token
        ngrok.set_auth_token(NGROK_AUTHTOKEN)
        print("✅ ngrok authenticated!")

        # Start Streamlit in background
        def run_streamlit():
            try:
                subprocess.run([
                    "streamlit", "run", STREAMLIT_APP,
                    "--server.port", "8501",
                    "--server.address", "localhost",
                    "--browser.serverAddress", "localhost"
                ])
            except Exception as e:
                print(f"❌ Streamlit error: {e}")

        # Start Streamlit first
        print("🚀 Starting Streamlit app...")
        streamlit_thread = threading.Thread(target=run_streamlit)
        streamlit_thread.daemon = True
        streamlit_thread.start()

        # Wait a moment for Streamlit to start
        time.sleep(3)

        # Start ngrok tunnel
        public_url = ngrok.connect(8501, bind_tls=True)
        print(f"🌐 Public URL: {public_url}")
        print(f"📱 Mobile QR: https://api.qrserver.com/v1/create-qr-code/?size=200x200&data={public_url}")

        print("\n📋 Your app is now publicly accessible!")
        print("Press Ctrl+C to stop the application")

        # Keep running
        try:
            while True:
                time.sleep(1)
        except KeyboardInterrupt:
            print("\n🛑 Shutting down...")
            ngrok.disconnect(public_url)
            ngrok.kill()

    except Exception as e:
        print(f"❌ Error: {e}")
        print("💡 Make sure you:")
        print("   - Have a valid ngrok auth token")
        print("   - Installed required packages: pip install pyngrok streamlit")
        print("   - Have your Streamlit app file in the same directory")

if __name__ == "__main__":
    setup_ngrok()

✅ ngrok authenticated!
🚀 Starting Streamlit app...


🌐 Public URL: NgrokTunnel: "https://af1ebc50e8e3.ngrok-free.app" -> "http://localhost:8501"
📱 Mobile QR: https://api.qrserver.com/v1/create-qr-code/?size=200x200&data=NgrokTunnel: "https://af1ebc50e8e3.ngrok-free.app" -> "http://localhost:8501"

📋 Your app is now publicly accessible!
Press Ctrl+C to stop the application


In [ ]:
# First, install and setup Java & Spark properly
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Then initialize Spark
from pyspark.sql import SparkSession

def init_spark():
    spark = SparkSession.builder \
        .appName("MLModelDeployment") \
        .config("spark.driver.memory", "2g") \
        .config("spark.executor.memory", "2g") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .getOrCreate()
    return spark

spark = init_spark()